# 02 · Rainfall time series, reference comparison and manuscript validation

**Depends on 01.** Demonstration time series plot is created from the output selected in notebook 01. An optional reproduction test compares original full-history retrieval against the published NetCDF by exact link ID/time/finite flags; this has already passed on one real HPC link. Optional gauge validation runs the original manuscript plotter. Its exact historical detector has not been found; selecting compatible v3 is an **explicit unverified experiment**, not the official historical replay.

**About this notebook.** Needs: the registered rainfall and the gauge file. Parameters: `RUN_FULL_HISTORY_COMPARISON`, `RUN_GAUGE_VALIDATION`, `ALLOW_UNVERIFIED_V3` (all off by default). Makes: a rainfall time series figure, an optional one link numerical regression, optional gauge validation figures. Cost: seconds by default; the optional steps are heavy. Fidelity: the one link regression is exact; the gauge validation uses a substitute detector and its numbers are not confirmed identical to the paper. Details: `docs/NOTEBOOK_GUIDE.md`.

In [1]:
from pathlib import Path
import sys, json
# One shared configuration for ALL notebooks (no per-notebook path editing).
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
assert (ROOT/'tools_verify_originals.py').is_file(), f'Wrong notebook working folder: {ROOT}'
sys.path.insert(0,str(ROOT/'src'))
from essd_cz.connected import Project
SETTINGS=json.loads((ROOT/'config/active_profile.json').read_text(encoding='utf-8'))
PROFILE=SETTINGS.get('profile','hpc')
if PROFILE not in ('hpc','user'):raise ValueError("Profile must be 'hpc' or 'user'")
CONFIG=ROOT/('config/hpc_source.example.json' if PROFILE=='hpc' else 'config/user_paths.json')
WORKDIR=ROOT/SETTINGS.get('workspace','results/connected_workflow')
project=Project(CONFIG,WORKDIR)
print('Profile:',PROFILE,'Data config:',project.config,'Output:',project.output)


Profile: hpc Data config: /home/jovyan/ESSD_publication_dataset/ESSD_CZ_2021_CONNECTED_WALKTHROUGH_V2/config/hpc_source.example.json Output: /home/jovyan/ESSD_publication_dataset/ESSD_CZ_2021_CONNECTED_WALKTHROUGH_V2/results/connected_workflow


In [2]:
import xarray as xr, pandas as pd, numpy as np, matplotlib.pyplot as plt
from essd_cz.rainfall import normalize_id
RAIN = project.get('rainfall')
PLOT_CML_INDEX = 3
START = '2021-07-13 16:05:00'
END = '2021-07-14 00:00:00'
out = project.stage_dir('02_rainfall_figures')
with xr.open_dataset(RAIN) as ds:
    cid = normalize_id(ds.cml_id.values[PLOT_CML_INDEX])
    d = ds.sel(time_5min=slice(START,END)).isel(cml_id=PLOT_CML_INDEX)
    if d.sizes['time_5min'] == 0: raise ValueError('Chosen time window is missing')
    plt.figure(figsize=(11,3.7))
    for k, sub in enumerate(d.sublink_id.values):
        plt.plot(d.time_5min.values,d.R_sublink_5min.isel(sublink_id=k).values,lw=.8,label=f'Direction {normalize_id(sub)}')
    if 'R_mean_5min' in d:plt.plot(d.time_5min.values,d.R_mean_5min.values,lw=1.4,label='Mean of valid directions')
    plt.ylabel('5-min CML rain rate (mm/h)');plt.xlabel('Time (UTC)');plt.title('Review timeline: '+cid);plt.legend();plt.grid(alpha=.2);plt.tight_layout()
    result=out/'reviewer_cml_rain_timeseries.png';plt.savefig(result,dpi=180);plt.close()
project.register('rain_timeline',result,'direct visualization of stage-01 5-min rain, not manuscript event benchmark')

[REGISTERED] rain_timeline: /home/jovyan/ESSD_publication_dataset/ESSD_CZ_2021_CONNECTED_WALKTHROUGH_V2/results/connected_workflow/02_rainfall_figures/reviewer_cml_rain_timeseries.png
  source: direct visualization of stage-01 5-min rain, not manuscript event benchmark


PosixPath('/home/jovyan/ESSD_publication_dataset/ESSD_CZ_2021_CONNECTED_WALKTHROUGH_V2/results/connected_workflow/02_rainfall_figures/reviewer_cml_rain_timeseries.png')

In [3]:
# Optional exact numerical regression. Keep the notebook kernel independent
# of the compatible scientific Python environment.
import subprocess

RUN_FULL_HISTORY_COMPARISON = False  # change to True when ready; full QC history is processed per link
LINK_INDICES = [3]                   # physical QC link indices; add others if needed
COMPARISON_START = "2021-04-01"
COMPARISON_DAYS = 14
NUMBA_THREADS = 2

# One interpreter setting shared by HPC and external users.
SCIENCE_PYTHON = (SETTINGS.get("science_python_by_profile") or {}).get(PROFILE)
if not SCIENCE_PYTHON:
    # Set science_python_by_profile in config/active_profile.json instead of
    # editing this notebook; the notebook kernel's interpreter is the fallback.
    SCIENCE_PYTHON = sys.executable
if RUN_FULL_HISTORY_COMPARISON:
    stage_output = project.stage_dir("02_numeric_comparison")
    cmd = [
        str(SCIENCE_PYTHON), str(ROOT / "run.py"),
        "--config", str(CONFIG),
        "--output-dir", str(stage_output),
        "compare-rainfall",
        "--start", COMPARISON_START,
        "--days", str(COMPARISON_DAYS),
        "--threads", str(NUMBA_THREADS),
        "--atol", "0.0001",
        "--rtol", "0.0001",
    ]
    for idx in LINK_INDICES:
        cmd.extend(["--link-index", str(idx)])
    print("Running verified comparison CLI:", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)
    report_file = stage_output / "rainfall_comparison" / "comparison_report.json"
    comparison_report = json.loads(report_file.read_text(encoding="utf-8"))
    print("Reference comparison all passed:", comparison_report["all_pass"])
else:
    print("Skipped optional full-history numerical comparison. Set RUN_FULL_HISTORY_COMPARISON=True to run it.")


Skipped optional full-history numerical comparison. Set RUN_FULL_HISTORY_COMPARISON=True to run it.


In [5]:
# Optional original gauge validation: confusion, wet-only hexbin and event.
from essd_cz.workflows import validation
RUN_GAUGE_VALIDATION = False  # computationally heavy; original publication detector unresolved
ALLOW_UNVERIFIED_V3 = False  # explicit opt-in only; this v3 is NOT proven historical source
EXACT_DETECTOR = None
PREFLIGHT_ONLY = False
if RUN_GAUGE_VALIDATION:
    figs=validation({**project.paths,'rain':project.get('rainfall')},project.stage_dir('02_gauge_validation'),
      detector=EXACT_DETECTOR,allow_unverified_v3=ALLOW_UNVERIFIED_V3,
      skip_event=True,only_check_sources=PREFLIGHT_ONLY)
    print('Gauge validation output:',figs)
else:
    print('Time-series figure saved. Gauge validation is an optional original-source step.')

ORIGINAL COMMAND: /home/jovyan/.venvs/pr13_xarray_2026_04/bin/python /home/jovyan/ESSD_publication_dataset/ESSD_CZ_2021_CONNECTED_WALKTHROUGH_V2/original/code/figures/rebuild_ESSD_FINAL_GRAF_shared_population_NEW_EVENT_FINAL_HEXBIN_SPACING.py --root /home/jovyan/ESSD_publication_dataset/ESSD_CZ_2021_CONNECTED_WALKTHROUGH_V2/results/connected_workflow/02_gauge_validation/validation_workspace --graf-rain-nc /home/jovyan/ESSD_publication_dataset/ESSD_CZ_2021_CONNECTED_WALKTHROUGH_V2/results/connected_workflow/01_rainfall_rebuild/recomputed_CZ_CML_RAINRATE_GRAF_ITU2005_PASTOREK_5MIN.nc --skip-event
FINAL GRAF RAIN SOURCE: /home/jovyan/ESSD_publication_dataset/ESSD_CZ_2021_CONNECTED_WALKTHROUGH_V2/results/connected_workflow/01_rainfall_rebuild/recomputed_CZ_CML_RAINRATE_GRAF_ITU2005_PASTOREK_5MIN.nc
  cml_id count: 2219; rainfall vars: ['R_sublink_5min']
cml: /home/jovyan/ESSD_publication_dataset/ESSD_CZ_2021_CONNECTED_WALKTHROUGH_V2/results/connected_workflow/02_gauge_validation/validation

/home/jovyan/.venvs/pr13_xarray_2026_04/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,


Gauge validated 150/1144 midpoint-matched physical CMLs
Gauge validated 300/1144 midpoint-matched physical CMLs
Gauge validated 450/1144 midpoint-matched physical CMLs
Gauge validated 600/1144 midpoint-matched physical CMLs
Gauge validated 750/1144 midpoint-matched physical CMLs
Gauge validated 900/1144 midpoint-matched physical CMLs
Gauge validated 1050/1144 midpoint-matched physical CMLs
SAVED /home/jovyan/ESSD_publication_dataset/ESSD_CZ_2021_CONNECTED_WALKTHROUGH_V2/results/connected_workflow/02_gauge_validation/validation_workspace/figures/FINAL_GRAF_hexbin_gauge_202104_202110.png and PDF
SAVED /home/jovyan/ESSD_publication_dataset/ESSD_CZ_2021_CONNECTED_WALKTHROUGH_V2/results/connected_workflow/02_gauge_validation/validation_workspace/figures/FINAL_GRAF_hexbin_gauge_WETONLY_202104_202110.png and PDF
SAVED /home/jovyan/ESSD_publication_dataset/ESSD_CZ_2021_CONNECTED_WALKTHROUGH_V2/results/connected_workflow/02_gauge_validation/validation_workspace/figures/FINAL_GRAF_confusion_GAUG